# GSD–PI — Algorithm 1: Manuscript-aligned review version (v2)

**Purpose:** Run the topology-only selection policy on the original active-source entity tables, then assess attack labels *post hoc*. This notebook corrects the v1 decision order for weak and intermediate MGR regimes.

**Important distinction:** When `TC=0` and `MGR<2`, P95 is retained *only as a descriptive reference*, not an endorsed detection candidate. When `2 ≤ MGR < 5`, the policy abstains even if `TC=0`. For `MGR≥5`, TC determines P95 vs admissible gap recovery.

**Input:** `entity_table_NF-ToN-IoT-v2 (1).csv`, `entity_table_NF-CSE-CIC-IDS2018-v2.csv` (source-level tables; do not publish them if source keys identify IP addresses). Place in `/content` using Google Colab or change `INPUT_DIR`.

**Outputs:** three aggregate-only CSV reports: `GSD_PI_Audit_v2.csv`, `GSD_PI_Baselines_v2.csv`, and `GSD_PI_Sensitivity_v2.csv`. No individual source/IP identifiers are written.

**Scientific status:** Corrected code + synthetic-unit-test checks. **Not yet a fresh execution on the original real datasets.** Verify observed outputs with the revised manuscript before archival release.


In [1]:
# In Google Colab only: uncomment to upload the two input CSV files
# from google.colab import files
# files.upload()


In [3]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

root = Path('/content/drive/MyDrive')

required = {
    'entity_table_NF-ToN-IoT-v2 (1).csv':
        'entity_table_NF-ToN-IoT-v2*.csv',
    'entity_table_NF-CSE-CIC-IDS2018-v2.csv':
        'entity_table_NF-CSE-CIC-IDS2018-v2*.csv'
}

for destination, pattern in required.items():
    matches = list(root.rglob(pattern))
    if not matches:
        print(f'NOT FOUND: {pattern}')
        continue

    if len(matches) > 1:
        print(f'MULTIPLE MATCHES for {pattern}:')
        for p in matches:
            print(' ', p)
        continue

    shutil.copy2(matches[0], Path('/content') / destination)
    print(f'READY: {destination}')

print('Data preparation completed.')


Mounted at /content/drive
READY: entity_table_NF-ToN-IoT-v2 (1).csv
READY: entity_table_NF-CSE-CIC-IDS2018-v2.csv
Data preparation completed.


In [4]:
"""GSD–PI Algorithm 1: reproducible degree-based audit (manuscript-aligned v2).

Python 3.10+, numpy. Reads public benchmark source-level aggregate tables.
Inputs: entity_table_NF-ToN-IoT-v2 (1).csv and
        entity_table_NF-CSE-CIC-IDS2018-v2.csv
No individual IP/source identifiers are written to outputs.

IMPORTANT: Proposed algorithm/heuristics pending external validation; empirical
attack-selectivity is NOT predictable from unlabeled degree data alone.
"""
from pathlib import Path
import csv, math
import numpy as np

# Edit INPUT_DIR for Colab/Drive as required. CSV files can be in /content.
INPUT_DIR = Path('/content') if Path('/content/entity_table_NF-CSE-CIC-IDS2018-v2.csv').exists() else Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
OUTPUT_DIR = INPUT_DIR
INPUTS = {
    'NF-ToN-IoT-v2': 'entity_table_NF-ToN-IoT-v2 (1).csv',
    'NF-CSE-CIC-IDS2018-v2': 'entity_table_NF-CSE-CIC-IDS2018-v2.csv',
}
# Original-paper cutoffs are retained for an AUDIT, not claimed validated.
DELTA_TC = 0.05
GAMMA_MIN = 3.0
K_MIN = 2
TOP_K = 30
MGR_RECOVERY_MIN = 5.0   # provisional original-paper regime heuristic
MGR_HOMOGENEOUS_MAX = 2.0  # descriptor ONLY; no attack-utility assertion


def load_entities(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing {path}; upload it to Colab or modify INPUT_DIR')
    degrees, labels = [], []
    seen = set()
    with path.open(encoding='utf-8-sig', newline='') as fh:
        reader = csv.DictReader(fh)
        assert {'src_key', 'out_degree', 'src_is_attacker'}.issubset(reader.fieldnames or []), (path.name, reader.fieldnames)
        for row in reader:
            # Source identifiers are checked only and are never exported.
            source = row['src_key']
            if source in seen:
                raise ValueError(f'Duplicate source entity in {path.name}; aggregate before evaluating')
            seen.add(source)
            d = int(row['out_degree'])
            if d <= 0:
                raise ValueError('Input must contain ACTIVE SOURCE entities with degree >= 1')
            value = str(row['src_is_attacker']).strip().lower()
            if value not in ('0', '1', 'false', 'true'):
                raise ValueError('Expected binary source-level ground truth (0/1)')
            degrees.append(d)
            labels.append(value in ('1', 'true'))
    return np.asarray(degrees, dtype=np.int64), np.asarray(labels, dtype=bool)


def gini_degree(degrees):
    x = np.sort(degrees.astype(np.float64))
    if not len(x) or x.sum() == 0:
        return math.nan
    return float(2 * np.dot(np.arange(1, len(x)+1), x)/(len(x)*x.sum()) - (len(x)+1)/len(x))


def wilson_interval(success, total, z=1.959963984540054):
    if total == 0:
        return math.nan, math.nan
    p = success / total
    a = 1 + z*z/total
    center = (p+z*z/(2*total))/a
    h = z*math.sqrt(p*(1-p)/total + z*z/(4*total*total))/a
    return center-h, center+h


def metrics(mask, y):
    n = len(y)
    selected = int(mask.sum())
    tp = int(np.count_nonzero(mask & y))
    fp = selected-tp
    attackers = int(y.sum())
    p = tp/selected if selected else math.nan
    base = attackers/n if n else math.nan
    ci_low, ci_high = wilson_interval(tp, selected)
    return dict(selected=selected, TP=tp, FP=fp, FN=attackers-tp,
                precision=p, recall=tp/attackers if attackers else math.nan,
                lift=p/base if selected and base else math.nan,
                precision_ci_low=ci_low, precision_ci_high=ci_high,
                tail_density_sources=selected/n if n else math.nan)


def topk_summary(degrees, labels, k):
    """Report precision bounds if a tied degree crosses the Top-k boundary."""
    n = len(degrees)
    if n == 0:
        return None
    k = min(k,n)
    cutoff = int(np.partition(degrees,n-k)[n-k])
    strictly = degrees>cutoff
    tie = degrees==cutoff
    remaining = k-int(strictly.sum())
    n_tie = int(tie.sum())
    n_tie_attacker = int((tie & labels).sum())
    tp_above = int((strictly & labels).sum())
    tp_min = tp_above+max(0,remaining-(n_tie-n_tie_attacker))
    tp_max = tp_above+min(remaining,n_tie_attacker)
    baseline = float(labels.mean())
    return dict(k=k, boundary_degree=cutoff, tie_boundary_ambiguous=bool(n_tie>remaining),
                TP_min=tp_min, TP_max=tp_max,
                precision_min=tp_min/k, precision_max=tp_max/k,
                lift_min=(tp_min/k)/baseline if baseline else math.nan,
                lift_max=(tp_max/k)/baseline if baseline else math.nan)


def protocol(degrees, *, delta_tc=DELTA_TC, gamma=GAMMA_MIN,
             k_min=K_MIN, top_k=TOP_K, mgr_recovery_min=MGR_RECOVERY_MIN):
    """Label-free protocol; no labels in signature or decisions.

    The paper's TC formula is preserved; cutoffs are exploratory heuristics.
    Returned status does NOT classify entities or topology as malicious/benign.
    """
    if not len(degrees) or (degrees<=0).any():
        raise ValueError('Expected nonempty positive-degree ACTIVE SOURCE vector')
    p90, p95, p97 = map(float, np.percentile(degrees, [90,95,97], method='linear'))
    tc = abs(p97-p90)/max(p97,1.0) < delta_tc
    td_p95 = float(np.mean(degrees>=p95))
    top = np.sort(degrees[degrees>1])[::-1][:top_k]
    ratios = top[:-1]/top[1:] if len(top)>1 else np.array([],dtype=float)
    mgr = float(np.max(ratios)) if len(ratios) else math.nan
    # Every significant gap i is between ranks i and i+1 (one-indexed).
    # A gap above threshold has no degree ties across the boundary.
    eligible = [(i+1, float(r), int(top[i])) for i,r in enumerate(ratios)
                if r>gamma and i+1>=k_min]
    # Avoid invalid gap outcomes: most extreme gap at rank 1 fails kmin=2.
    eligible.sort(key=lambda rec:rec[0])
    selected_gap_rank = None
    tau_gap = None
    # Decision order matches manuscript Algorithm 1 steps 5–10.
    # MGR isolation state is evaluated BEFORE the TC-dependent threshold branch.
    # A weak-tail, non-collapsed P95 is a descriptive reference only, not a strong candidate.
    if not np.isfinite(mgr):
        isolation_status = 'INCONCLUSIVE_INSUFFICIENT_TAIL'
        decision, tau = 'ABSTAIN_INSUFFICIENT_TAIL', None
    elif mgr < MGR_HOMOGENEOUS_MAX:
        isolation_status = 'WEAK_ISOLATION'
        if not tc:
            decision, tau = 'P95_REFERENCE_WEAK', p95
        else:
            decision, tau = 'ABSTAIN_WEAK_COLLAPSE', None
    elif mgr < mgr_recovery_min:
        isolation_status = 'INCONCLUSIVE_BOUNDARY'
        decision, tau = 'ABSTAIN_BOUNDARY', None
    else:
        isolation_status = 'STRONG_ISOLATION_CANDIDATE'
        if not tc:
            decision, tau = 'P95_STRONG_CANDIDATE', p95
        elif eligible:
            decision = 'GAP_RECOVERY_CANDIDATE'
            selected_gap_rank, chosen_gap_ratio, tau_gap = eligible[0]
            tau = float(tau_gap)
        else:
            decision, tau = 'ABSTAIN_NO_VALID_GAP', None
    selected_mask = (degrees>=tau) if tau is not None else np.zeros(len(degrees),dtype=bool)
    if tau is not None and (not selected_mask.any() or selected_mask.all()):
        # Even a descriptive P95 selection must not be misreported as effective triage
        # when it is empty or selects all eligible sources.
        decision, tau = 'ABSTAIN_EMPTY_OR_SATURATED', None
        selected_mask = np.zeros(len(degrees),dtype=bool)
    descriptive_reference_only = (decision == 'P95_REFERENCE_WEAK')
    # This selects only candidates for assessment; no pre-deployment assertion
    # of attack-related PI or broad graph-based IDS applicability is made.
    return dict(n_source_entities=int(len(degrees)), p90=p90, p95=p95, p97=p97,
                median_source_degree=float(np.median(degrees)),
                cv_source_degree=float(np.std(degrees,ddof=0)/np.mean(degrees)),
                gini_source_degree=gini_degree(degrees),
                p95_to_median=float(p95/np.median(degrees)),
                TC=bool(tc), TD_p95=td_p95,
                MGR_topk_non_singleton=mgr,
                valid_gap_ranks=';'.join(str(t[0]) for t in eligible),
                selected_gap_rank=selected_gap_rank,
                tau_gap=tau_gap,
                isolation_status=isolation_status,
                descriptive_reference_only=descriptive_reference_only,
                decision=decision, threshold=tau, mask=selected_mask)


def write_csv(path, rows):
    if not rows:
        return
    fields=list(rows[0].keys())
    with path.open('w',newline='',encoding='utf-8') as fh:
        writer=csv.DictWriter(fh,fieldnames=fields)
        writer.writeheader();writer.writerows(rows)


def run():
    output, baselines, sensitivity = [], [], []
    for dataset, filename in INPUTS.items():
        x, y = load_entities(INPUT_DIR/filename)
        d = protocol(x)
        m = metrics(d.pop('mask'),y)
        output.append(dict(dataset=dataset, n_attack_sources=int(y.sum()),
                           attack_base_rate=float(np.mean(y)),**d,**m))
        for name, threshold in [('P95',d['p95']),('P97',d['p97'])]:
            baselines.append(dict(dataset=dataset,method=name,threshold=threshold,
                                  **metrics(x>=threshold,y)))
        for k in (11,20,50,100):
            baseline = topk_summary(x,y,k)
            baselines.append(dict(dataset=dataset,method=f'Top-{k}',threshold=baseline['boundary_degree'],
                                  selected=k, TP_min=baseline['TP_min'],TP_max=baseline['TP_max'],
                                  precision_min=baseline['precision_min'],precision_max=baseline['precision_max'],
                                  lift_min=baseline['lift_min'],lift_max=baseline['lift_max'],
                                  tie_boundary_ambiguous=baseline['tie_boundary_ambiguous']))
        for gamma in (2.0,3.0,4.0,5.0):
            for kmin in (2,3,5):
                z=protocol(x,gamma=gamma,k_min=kmin)
                zmetrics=metrics(z['mask'],y) if z['threshold'] is not None else {}
                sensitivity.append(dict(dataset=dataset,gamma_min=gamma,k_min=kmin,
                    decision=z['decision'],valid_gap_ranks=z['valid_gap_ranks'],
                    threshold=z['threshold'],selected=zmetrics.get('selected'),
                    precision=zmetrics.get('precision'),lift=zmetrics.get('lift')))
        print(f"{dataset}: n={len(x):,}, attackers={y.sum():,}, TC={d['TC']}, "
              f"MGR={d['MGR_topk_non_singleton']:.3f}, decision={d['decision']}, "
              f"threshold={d['threshold']}, selected={m['selected']}, "
              f"TP={m['TP']}, P={m['precision']:.5f}, lift={m['lift']:.2f}")
    write_csv(OUTPUT_DIR/'GSD_PI_Audit_v2.csv',output)
    # Mixed baselines can contain different columns; union keys explicitly.
    all_fields = list(dict.fromkeys(key for row in baselines for key in row))
    with (OUTPUT_DIR/'GSD_PI_Baselines_v2.csv').open('w', newline='', encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=all_fields);w.writeheader();w.writerows(baselines)
    write_csv(OUTPUT_DIR/'GSD_PI_Sensitivity_v2.csv',sensitivity)
    print('Outputs written to',OUTPUT_DIR)
    return output,baselines,sensitivity


def _unit_tests():
    # Valid gap at 2 -> 3 (degree 100 -> degree 10): 2 selected, not 1.
    a=np.asarray([100,100,10,10]+[1]*996)
    z=protocol(a,top_k=10)
    assert z['decision']=='GAP_RECOVERY_CANDIDATE' and z['selected_gap_rank']==2 and z['mask'].sum()==2
    # A dramatic first gap alone is invalid for kmin=2: no forced selection of 1.
    b=np.asarray([1000,10,9,8]+[1]*996)
    w=protocol(b)
    assert w['TC'] and w['decision']=='ABSTAIN_NO_VALID_GAP' and w['threshold'] is None
    # A non-collapsed weak tail retains P95 only as a descriptive reference.
    c=np.asarray(list(range(1,101)))
    v=protocol(c)
    assert not v['TC'] and v['decision']=='P95_REFERENCE_WEAK'
    assert v['threshold']==v['p95'] and v['descriptive_reference_only']
    # The intermediate MGR band requires abstention even when TC=0.
    d=np.asarray(list(range(10,110))+[150,400])
    z=protocol(d)
    assert not z['TC'] and 2 <= z['MGR_topk_non_singleton'] < 5, z
    assert z['decision']=='ABSTAIN_BOUNDARY' and z['threshold'] is None
    # Non-collapsed strong tails select the P95 candidate, not an ad hoc gap.
    e=np.asarray(list(range(10,110))+[1000,1000])
    u=protocol(e)
    assert not u['TC'] and u['MGR_topk_non_singleton']>=5, u
    assert u['decision']=='P95_STRONG_CANDIDATE' and not u['descriptive_reference_only']
    # No false 'perfect precision' when zero selected.
    assert math.isnan(metrics(np.zeros(5,dtype=bool),np.asarray([True,False,False,False,False]))['precision'])
    print('Unit tests passed')

if __name__=='__main__':
    _unit_tests()
    run()


Unit tests passed
NF-ToN-IoT-v2: n=23,079, attackers=31, TC=True, MGR=12.466, decision=GAP_RECOVERY_CANDIDATE, threshold=214.0, selected=11, TP=10, P=0.90909, lift=676.80
NF-CSE-CIC-IDS2018-v2: n=224,568, attackers=648, TC=False, MGR=1.186, decision=P95_REFERENCE_WEAK, threshold=22.0, selected=11571, TP=301, P=0.02601, lift=9.02
Outputs written to /content


In [5]:

from google.colab import files

for filename in [
    "GSD_PI_Audit_v2.csv",
    "GSD_PI_Baselines_v2.csv",
    "GSD_PI_Sensitivity_v2.csv"
]:
    files.download("/content/" + filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reporting and verification

- `GSD_PI_Audit_v2.csv`: one row per full-source benchmark, including topology-only decision, isolation state, candidate/reference count, post-hoc metrics and Wilson intervals.
- `GSD_PI_Baselines_v2.csv`: retrospective P95/P97 and tie-aware Top-k comparisons.
- `GSD_PI_Sensitivity_v2.csv`: exploratory gamma and k-min perturbations; never used to re-tune the frozen policy.

**Check the expected results from the revised manuscript:** NF-ToN-IoT-v2: 23,079 active sources, 31 attack-labeled, TC=1, MGR ≈ 12.47, gap threshold 214, selected=11, TP=10. NF-CSE-CIC-IDS2018-v2: 224,568 active sources, 648 attack-labeled, TC=0, weak isolation MGR ≈ 1.19, P95 **reference** threshold 22, selected=11,571, TP=301. The latter is a descriptive low-selectivity comparison, not evidence of strong isolation.

If outputs diverge, **stop and investigate** before changing the manuscript or tagging a GitHub release.
